## SQL setup from csv file 

In [10]:
import os
import sqlite3
import pandas as pd

# Project root (parent of 'notebooks' folder)
BASE_DIR = os.path.dirname(os.getcwd())

# Common paths
DATA_DIR = os.path.join(BASE_DIR, "data")
DB_PATH = os.path.join(DATA_DIR, "credit.db")
CSV_PATH = os.path.join(DATA_DIR, "cleaned_data.csv")

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"DB_PATH:  {DB_PATH}")
print(f"CSV_PATH: {CSV_PATH}")

BASE_DIR: c:\Users\ASUS\Projectsme\sql-chat-agent
DATA_DIR: c:\Users\ASUS\Projectsme\sql-chat-agent\data
DB_PATH:  c:\Users\ASUS\Projectsme\sql-chat-agent\data\credit.db
CSV_PATH: c:\Users\ASUS\Projectsme\sql-chat-agent\data\cleaned_data.csv


In [13]:
# Load and inspect cleaned data
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (149717, 12)

Columns:
['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'income_missing_flag']

First 5 rows:


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,income_missing_flag
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2,0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1,0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0,0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0,0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0,0


In [14]:
# 1. Load cleaned data
df = pd.read_csv(CSV_PATH)

# 2. Rename columns to SQL-friendly names
df = df.rename(columns={
    "SeriousDlqin2yrs": "is_default",
    "RevolvingUtilizationOfUnsecuredLines": "credit_utilization",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "past_due_30_59",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "open_credit_lines",
    "NumberOfTimes90DaysLate": "past_due_90",
    "NumberRealEstateLoansOrLines": "real_estate_loans",
    "NumberOfTime60-89DaysPastDueNotWorse": "past_due_60_89",
    "NumberOfDependents": "num_dependents",
    "income_missing_flag": "income_was_missing"   # ← yangi, oson nom
})

# 3. Add borrower_id
df.insert(0, "borrower_id", range(1, len(df) + 1))

# 4. Verify
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (149717, 13)

Columns: ['borrower_id', 'is_default', 'credit_utilization', 'age', 'past_due_30_59', 'debt_ratio', 'monthly_income', 'open_credit_lines', 'past_due_90', 'real_estate_loans', 'past_due_60_89', 'num_dependents', 'income_was_missing']


,borrower_id,is_default,credit_utilization,age,past_due_30_59,debt_ratio,monthly_income,open_credit_lines,past_due_90,real_estate_loans,past_due_60_89,num_dependents,income_was_missing
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2,0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1,0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0,0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0,0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0,0


In [15]:
import sqlite3

conn = sqlite3.connect(DB_PATH)

# Write to SQL
df.to_sql("borrowers", conn, if_exists="replace", index=False)

# Verify
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM borrowers")
count = cursor.fetchone()[0]
print(f"✅ Wrote {count} rows to 'borrowers' table")

cursor.execute("PRAGMA table_info(borrowers)")
schema = cursor.fetchall()
print("\nSchema:")
for col in schema:
    print(f"  {col[1]} ({col[2]})")

conn.close()

✅ Wrote 149717 rows to 'borrowers' table

Schema:
  borrower_id (INTEGER)
  is_default (INTEGER)
  credit_utilization (REAL)
  age (INTEGER)
  past_due_30_59 (INTEGER)
  debt_ratio (REAL)
  monthly_income (REAL)
  open_credit_lines (INTEGER)
  past_due_90 (INTEGER)
  real_estate_loans (INTEGER)
  past_due_60_89 (INTEGER)
  num_dependents (INTEGER)
  income_was_missing (INTEGER)


In [17]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(DB_PATH)

# Query 1: Default count
print("=== Query 1: Default distribution ===")
result = pd.read_sql("""
    SELECT 
        is_default, 
        COUNT(*) as count,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM borrowers), 2) as percentage
    FROM borrowers 
    GROUP BY is_default
""", conn)
print(result)

# Query 2: Age stats
print("\n=== Query 2: Age statistics ===")
result = pd.read_sql("""
    SELECT 
        MIN(age) as min_age,
        MAX(age) as max_age,
        ROUND(AVG(age), 1) as avg_age
    FROM borrowers
""", conn)
print(result)

# Query 3: Default rate by age group
print("\n=== Query 3: Default rate by age group ===")
result = pd.read_sql("""
    SELECT 
        CASE 
            WHEN age < 30 THEN '<30'
            WHEN age < 45 THEN '30-44'
            WHEN age < 60 THEN '45-59'
            ELSE '60+'
        END as age_group,
        COUNT(*) as total,
        SUM(is_default) as defaulted,
        ROUND(100.0 * SUM(is_default) / COUNT(*), 2) as default_rate_pct
    FROM borrowers
    GROUP BY age_group
    ORDER BY age_group
""", conn)
print(result)

# Query 4: High-risk borrowers
print("\n=== Query 4: Top 10 high-risk borrowers ===")
result = pd.read_sql("""
    SELECT 
        borrower_id, age, monthly_income,
        past_due_90, past_due_60_89, past_due_30_59,
        is_default
    FROM borrowers
    WHERE past_due_90 > 0
    ORDER BY past_due_90 DESC, past_due_60_89 DESC
    LIMIT 10
""", conn)
print(result)

# Query 5: Income missing flag effect
print("\n=== Query 5: Default rate by income availability ===")
result = pd.read_sql("""
    SELECT 
        income_was_missing,
        COUNT(*) as count,
        ROUND(100.0 * SUM(is_default) / COUNT(*), 2) as default_rate_pct
    FROM borrowers
    GROUP BY income_was_missing
""", conn)
print(result)

conn.close()

=== Query 1: Default distribution ===
   is_default   count  percentage
0           0  139839        93.4
1           1    9878         6.6

=== Query 2: Age statistics ===
   min_age  max_age  avg_age
0       21       99     52.3

=== Query 3: Default rate by age group ===
  age_group  total  defaulted  default_rate_pct
0     30-44  38918       3656              9.39
1     45-59  53831       3761              6.99
2       60+  48291       1488              3.08
3       <30   8677        973             11.21

=== Query 4: Top 10 high-risk borrowers ===
   borrower_id  age  monthly_income  past_due_90  past_due_60_89  \
0       103135   62          2700.0           17               0   
1        30614   32          3879.0           15               0   
2       148821   34          3000.0           15               0   
3        89548   42          5560.0           14               1   
4       126443   38          4500.0           14               1   
5       137049   45          654

In [11]:
# Read CSV
df = pd.read_csv(CSV_PATH)

# Connect to DB
conn = sqlite3.connect(DB_PATH)